# r/wallstreetbets comment word frequency
**Problem:** Which tickers and themes dominate the discussion on a busy subreddit right now?
**Approach:** Fetch the 10 hottest r/wallstreetbets posts with PRAW and gather the top-level comments of the first one (the daily discussion thread). Tokenise with NLTK, strip English + custom WSB stop words (`stop_words_addon.py`), rank the rest with `FreqDist`, then POS-tag and named-entity-chunk the top terms.
**Data:** Public Reddit data, read-only. Outputs are cleared here, so run it with your own app credentials.
**Credentials:** supplied by you through environment variables. None are stored in this repo. Copy `.env.example`, fill it in, and export the variables before starting Jupyter:
```bash
# macOS / Linux
export REDDIT_CLIENT_ID="..."
# Windows PowerShell
$env:REDDIT_CLIENT_ID = "..."
```

In [ ]:
import os
import praw
import pandas as pd
import datetime as dt

import stop_words_addon as swta

import nltk
from nltk.corpus import stopwords
from nltk import FreqDist

In [ ]:
# Read-only access needs only an app ID/secret and a user agent (see .env.example):
#   export REDDIT_CLIENT_ID=...  REDDIT_CLIENT_SECRET=...  REDDIT_USER_AGENT="wsb-word-freq by <your-username>"
# (Add username=/password= from your own env vars only if you need authenticated actions.)
reddit = praw.Reddit(client_id=os.environ["REDDIT_CLIENT_ID"],
                     client_secret=os.environ["REDDIT_CLIENT_SECRET"],
                     user_agent=os.environ["REDDIT_USER_AGENT"])

In [ ]:
subreddit = reddit.subreddit("wallstreetbets")  # any public subreddit works, e.g. "CryptoCurrency"

In [ ]:
hot_subreddit = subreddit.hot(limit=10)

In [ ]:
topics_dict = { "title":[],
               "score":[],
               "id":[],
               "url":[],
               "comms_num":[],
               "created":[],
               "body":[]}

In [ ]:
for submission in hot_subreddit:
    topics_dict["title"].append(submission.title)
    topics_dict["score"].append(submission.score)
    topics_dict["id"].append(submission.id)
    topics_dict["url"].append(submission.url)
    topics_dict["comms_num"].append(submission.num_comments)
    topics_dict["created"].append(submission.created)
    topics_dict["body"].append(submission.selftext)

In [ ]:
topics_data = pd.DataFrame(topics_dict)

In [ ]:
def get_date(created):
    return dt.datetime.fromtimestamp(created)

In [ ]:
_timestamp = topics_data["created"].apply(get_date)

In [ ]:
topics_data = topics_data.assign(timestamp = _timestamp)

In [ ]:
topics_data

In [ ]:
topics_data["title"][0]

In [ ]:
post_id = topics_data["id"][0]  # first hot post (usually the daily discussion thread)

In [ ]:
submission = reddit.submission(id=post_id)

In [ ]:
sentence = ""

In [ ]:
submission.comments.replace_more(limit=0)  # drop 'load more comments' placeholders (they have no .body)
for top_level_comment in submission.comments:
    sentence = sentence + " " + top_level_comment.body

In [ ]:
tokens0 = nltk.word_tokenize(sentence)

In [ ]:
tokens1 = [word for word in tokens0 if word.isalpha()]

In [ ]:
tokens2 = [w.lower() for w in tokens1]

In [ ]:
stop_words = stopwords.words('english')

In [ ]:
stop_words = swta.add_stop_words(stop_words)

In [ ]:
# filter out stop words
tokens3 = [w for w in tokens2 if not w in stop_words]

In [ ]:
print(len(tokens0))
print(len(tokens1))
print(len(tokens2))
print(len(tokens3))

In [ ]:
freq_dist_pos = FreqDist(tokens3)
i = 0
for items in freq_dist_pos.most_common(1000):
    print(i, " : ", items)
    i=i+1

In [ ]:
# paste-ready candidates for extending stop_words_addon.py
for element in freq_dist_pos:
    print("stop_words.append('"+element+"')")

In [ ]:
freqI = min(30, len(freq_dist_pos))  # top-N terms to analyse

In [ ]:
test = freq_dist_pos.most_common(freqI)

In [ ]:

listFreq = []
listValue = []
i =0
while i < freqI:
    listFreq.append(test[i][0])
    listValue.append(test[i][1])
    i=i+1

In [ ]:
total = sum(listValue[:5])

In [ ]:
# share of the single most frequent term among the top 5 (%)
listValue[0] / total *100

In [ ]:
tagged = nltk.pos_tag(listFreq)

In [ ]:
entities = nltk.chunk.ne_chunk(tagged[:5])
print(entities)  # use entities.draw() for a tree window in a desktop session